In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
from torch import tensor
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
BATCH_SIZE=64
epochs=300
EPOCHS_PER_EVAL=10
lr=1e-2
hidden_features=512

In [ ]:
df = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2)
train_df = train_df.to_numpy()
test_df = test_df.to_numpy()

y_train, x_train = train_df[:,0], train_df[:,1:].astype(np.float32)
y_test, x_test = test_df[:,0], test_df[:,1:].astype(np.float32)

x_train /= 255.0
x_test  /= 255.0

In [ ]:
x_train, y_train, x_test, y_test = map(torch.tensor, [x_train, y_train, x_test, y_test])

In [ ]:
x_train, y_train, x_test, y_test = [t.to(device) for t in [x_train, y_train, x_test, y_test]]

In [ ]:
w1 = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((hidden_features, 784)).to(device).requires_grad_()
b1 = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((hidden_features,)).to(device).requires_grad_()
w2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((hidden_features, hidden_features)).to(device).requires_grad_()
b2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((hidden_features,)).to(device).requires_grad_()
w3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((10, hidden_features)).to(device).requires_grad_()
b3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((10, )).to(device).requires_grad_()

params = [w1, w2, w3, b1, b2, b3]

In [ ]:
class MNIST(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.n = len(x)
    def __len__(self):
        return self.n
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [ ]:
train_df = MNIST(x_train, y_train)
test_df = MNIST(x_test, y_test)
train_dl = DataLoader(train_df, batch_size=BATCH_SIZE)
test_dl = DataLoader(test_df, batch_size=BATCH_SIZE)

In [ ]:
def cross_entropy_loss(logits, y):
    logprob = torch.log_softmax(logits, dim=1)
    loss = -logprob[torch.arange(len(y)), y].mean()
    return loss

In [ ]:
def relu(x):
    return x.clip(min=0)
def mult(x, w, b):
    return x @ w.t() + b

def forward(x):
    a1 = relu(mult(x, w1, b1))
    a2 = relu(mult(a1, w2, b2))
    z3 = mult(a2, w3, b3)
    return z3

In [ ]:
def accuracy(x, y) :
    probs = forward(x)
    ypred = torch.argmax(probs, -1)
    return (y == ypred).to(float).mean()

In [ ]:
for i in range(epochs) :
    for x, y in train_dl :
        y_pred = forward(x)
        loss = cross_entropy_loss(y_pred, y)
        loss.backward()
        with torch.no_grad():
            for j in range(len(params)):
                params[j] -= lr * params[j].grad
        for j in range(len(params)) :
            params[j].grad.zero_()
    if (i + 1) % EPOCHS_PER_EVAL == 0 :
        print(f"Epoch {i + 1} Loss = {loss.item()}, Accuracy : {accuracy(x_test, y_test)}")